In [1]:
import pandas as pd
import numpy as np

In [2]:
import os
os.getcwd()
os.chdir('a:\\Dominic\\My Projects\\Home Credit\\')

In [62]:
Payments = pd.read_csv('data/Internal/installments_payments.csv')

In [63]:
Payments.shape

(13605401, 8)

In [5]:
Payments.dtypes

SK_ID_PREV                  int64
SK_ID_CURR                  int64
NUM_INSTALMENT_VERSION    float64
NUM_INSTALMENT_NUMBER       int64
DAYS_INSTALMENT           float64
DAYS_ENTRY_PAYMENT        float64
AMT_INSTALMENT            float64
AMT_PAYMENT               float64
dtype: object

In [64]:
Payments = Payments.astype({'SK_ID_PREV': 'string', 'SK_ID_CURR': 'string'})

In [50]:
base = pd.read_csv('data/Internal/application_train.csv', usecols=['SK_ID_CURR', 'TARGET'])
base = base.astype({'SK_ID_CURR': 'string'})

In [65]:
Payments = base.merge(Payments, on='SK_ID_CURR', how='inner')

In [66]:
Payments['MONTHS_BALANCE'] = (Payments['DAYS_INSTALMENT'] / 30).astype('int')

In [67]:
Payments.sort_values(by=['SK_ID_CURR', 'SK_ID_PREV', 'MONTHS_BALANCE'], inplace=True)

In [68]:
Payments = Payments[Payments['MONTHS_BALANCE'] >= - 60]

Impute missing values

In [69]:
Payments['DAYS_ENTRY_PAYMENT'] = Payments.groupby('SK_ID_PREV')['DAYS_ENTRY_PAYMENT'].transform(
    lambda x: x.ffill(limit=6).bfill(limit=6)
) #.reset_index()
Payments['AMT_PAYMENT'] = Payments.groupby('SK_ID_PREV')['AMT_PAYMENT'].transform(
    lambda x: x.ffill(limit=6).bfill(limit=6)
) #.reset_index()

Check if `NUM_INSTALMENT_VERSION` keeps changing or not.

In [70]:
Payments['INSTALMENT_VERSION_CHANGE'] = Payments.groupby(['SK_ID_PREV'])['NUM_INSTALMENT_VERSION'].diff().fillna(0).astype('int')

Aditional attributes

In [71]:
Payments['LATE_PAYMENT_DAYS'] = Payments['DAYS_ENTRY_PAYMENT'] - Payments['DAYS_INSTALMENT']

Payments['INSTALLMENT_RATIO'] = np.where(Payments['AMT_INSTALMENT'] > 0 ,
                                         Payments['AMT_PAYMENT'] / Payments['AMT_INSTALMENT'] , 
                                         np.nan)
Payments['LATE_PAYMENT_FLAG'] = (Payments['LATE_PAYMENT_DAYS'] > 0).astype('int')

Monthly Data

In [83]:
sum_cols = [
    'AMT_PAYMENT',
    'AMT_INSTALMENT',
    'INSTALMENT_VERSION_CHANGE',
    'LATE_PAYMENT_FLAG'
]

In [84]:
Payments_Mon_Sum = Payments.groupby(['SK_ID_CURR','MONTHS_BALANCE'])[sum_cols].sum()

In [85]:
Payments_Mon_Sum.columns = ['PMNT_INSTLMT_' + ''.join(col).strip().upper() for col in Payments_Mon_Sum.columns.values]
Payments_Mon_Sum = Payments_Mon_Sum.reset_index()

In [86]:
Payments_Mon_Sum.columns

Index(['SK_ID_CURR', 'MONTHS_BALANCE', 'PMNT_INSTLMT_AMT_PAYMENT',
       'PMNT_INSTLMT_AMT_INSTALMENT', 'PMNT_INSTLMT_INSTALMENT_VERSION_CHANGE',
       'PMNT_INSTLMT_LATE_PAYMENT_FLAG'],
      dtype='object')

In [87]:
max_cols = [
    'LATE_PAYMENT_DAYS',
    'INSTALLMENT_RATIO'
]

In [88]:
Payments_Mon_Max = Payments.groupby(['SK_ID_CURR','MONTHS_BALANCE'])[max_cols].max()

In [89]:
Payments_Mon_Max.columns = ['PMNT_INSTLMT_' + ''.join(col).strip().upper() for col in Payments_Mon_Max.columns.values]
Payments_Mon_Max = Payments_Mon_Max.reset_index()

In [90]:
Payments_Mon_Max.columns

Index(['SK_ID_CURR', 'MONTHS_BALANCE', 'PMNT_INSTLMT_LATE_PAYMENT_DAYS',
       'PMNT_INSTLMT_INSTALLMENT_RATIO'],
      dtype='object')

In [91]:
Payments_Mon = Payments_Mon_Sum.merge(Payments_Mon_Max, on=['SK_ID_CURR','MONTHS_BALANCE'], how='left')

In [92]:
del Payments_Mon_Max, Payments_Mon_Sum

Other Aggregates at previous ID level

In [93]:
Payments_Agg_PrevID = Payments.groupby(['SK_ID_CURR', 'SK_ID_PREV']).agg(
    PMNT_INSTLMT_VRSN_CHANGES_COUNT = ('NUM_INSTALMENT_VERSION', lambda x: x.nunique() ), #number of unique installment version changes per previous application
    PMNT_INSTLMT_DURATION_DAYS = ('DAYS_ENTRY_PAYMENT', lambda x: x.max() - x.min() + 1), # duration of installment payments in days
    PMNT_INSTLMT_AVG_INSTALLMENT_FREQ = ('DAYS_ENTRY_PAYMENT', lambda x: pd.Series(x.unique()).sort_values().diff().mean() if len(x) > 1 else np.nan), # average frequency of installment payments in days
    PMNT_INSTLMT_NUM_INSTALMENTS = ('NUM_INSTALMENT_NUMBER', 'count'),
).reset_index()

In [96]:
Payments_Agg_PrevID_counts = Payments_Agg_PrevID.groupby('SK_ID_CURR').agg({
    'PMNT_INSTLMT_VRSN_CHANGES_COUNT': ['mean', 'max', 'min'],
    'PMNT_INSTLMT_DURATION_DAYS': ['mean', 'max', 'min'],
    'PMNT_INSTLMT_AVG_INSTALLMENT_FREQ': ['mean', 'max', 'min'],    
    'PMNT_INSTLMT_NUM_INSTALMENTS': ['mean', 'max', 'min']
})

In [97]:
Payments_Agg_PrevID_counts.columns = ['_'.join(col).strip().upper() for col in Payments_Agg_PrevID_counts.columns.values]
Payments_Agg_PrevID_counts = Payments_Agg_PrevID_counts.reset_index()

In [98]:
Payments_Agg_PrevID_Last1Y = Payments[Payments['MONTHS_BALANCE']>=-12].groupby(['SK_ID_CURR', 'SK_ID_PREV']).agg(
    PMNT_INSTLMT_VRSN_CHANGES_COUNT_L1Y = ('NUM_INSTALMENT_VERSION', lambda x: x.nunique() ), #number of unique installment version changes per previous application
    PMNT_INSTLMT_AVG_INSTALLMENT_FREQ_L1Y = ('DAYS_ENTRY_PAYMENT', lambda x: pd.Series(x.unique()).sort_values().diff().mean() if len(x) > 1 else np.nan), # average frequency of installment payments in days
).reset_index()

In [101]:
Payments_Agg_PrevID_Last1Y_counts = Payments_Agg_PrevID_Last1Y.groupby('SK_ID_CURR').agg({
    'PMNT_INSTLMT_VRSN_CHANGES_COUNT_L1Y': ['mean', 'max', 'min'],
    'PMNT_INSTLMT_AVG_INSTALLMENT_FREQ_L1Y': ['mean', 'max', 'min']
})

In [102]:
Payments_Agg_PrevID_Last1Y_counts.columns = ['_'.join(col).strip().upper() for col in Payments_Agg_PrevID_Last1Y_counts.columns.values]
Payments_Agg_PrevID_Last1Y_counts = Payments_Agg_PrevID_Last1Y_counts.reset_index()

In [103]:
del Payments_Agg_PrevID

`PMNT_AVG_INSTALLMENT_FREQ` the average days between payments, is negative in few cases. This is because in some IDs, the `DAYS_ENTRY_PAYMENT` is decreasng with Installment Number , ie higher installment number indicates an earlier payment rather than a later payment. `NUM_INSTALMENT_NUMBER` is not in correct order, need to sort data based on `DAYS_ENTRY_PAYMENT`.

**last 1 Year**

In [104]:
def aggregate_features(df, group_col, agg_cols, agg_funcs=('mean', 'max', 'min')):
    """
    Aggregates selected columns using specified functions.

    Args:
        df (pd.DataFrame): Input dataframe
        group_col (str): Column to group by (e.g., 'SK_ID_CURR')
        cols (list): List of columns to aggregate
        agg_funcs (tuple/list): Aggregations to apply

    Returns:
        pd.DataFrame: Aggregated dataframe with renamed columns
    """

    # Work on a copy to avoid modifying original data
    df = df[[group_col] + agg_cols].copy()

    # Perform aggregation
    agg_df = df.groupby(group_col)[agg_cols].agg(agg_funcs)

    # Reset index so it's merge-friendly
    agg_df = agg_df.reset_index()

    return agg_df

In [105]:
agg_cols = [
    'PMNT_INSTLMT_AMT_PAYMENT',
    'PMNT_INSTLMT_AMT_INSTALMENT', 
    'PMNT_INSTLMT_INSTALMENT_VERSION_CHANGE',
    'PMNT_INSTLMT_LATE_PAYMENT_DAYS', 
    'PMNT_INSTLMT_INSTALLMENT_RATIO',
    'PMNT_INSTLMT_LATE_PAYMENT_FLAG'
]

In [106]:
Payments_Mon.columns

Index(['SK_ID_CURR', 'MONTHS_BALANCE', 'PMNT_INSTLMT_AMT_PAYMENT',
       'PMNT_INSTLMT_AMT_INSTALMENT', 'PMNT_INSTLMT_INSTALMENT_VERSION_CHANGE',
       'PMNT_INSTLMT_LATE_PAYMENT_FLAG', 'PMNT_INSTLMT_LATE_PAYMENT_DAYS',
       'PMNT_INSTLMT_INSTALLMENT_RATIO'],
      dtype='object')

In [107]:
Payments_Mon_1Y_Avg = aggregate_features(Payments_Mon[Payments_Mon['MONTHS_BALANCE']>=-12], 'SK_ID_CURR', agg_cols, agg_funcs='mean')
Payments_Mon_1Y_Avg.columns = [f"{col}_1Y_AVG" if col != 'SK_ID_CURR'  else col for col in Payments_Mon_1Y_Avg.columns]

In [108]:
Payments_Mon_1Y_Max = aggregate_features(Payments_Mon[Payments_Mon['MONTHS_BALANCE']>=-12], 'SK_ID_CURR', agg_cols, agg_funcs='max')
Payments_Mon_1Y_Max.columns = [f"{col}_1Y_MAX" if col != 'SK_ID_CURR'  else col for col in Payments_Mon_1Y_Max.columns]

Last Quarter

In [109]:
Payments_Mon_3M_Avg = aggregate_features(Payments_Mon[Payments_Mon['MONTHS_BALANCE']>=-3], 'SK_ID_CURR', agg_cols, agg_funcs='mean')
Payments_Mon_3M_Avg.columns = [f"{col}_3M_AVG" if col != 'SK_ID_CURR'  else col for col in Payments_Mon_3M_Avg.columns]

In [110]:
Payments_Mon_3M_Max = aggregate_features(Payments_Mon[Payments_Mon['MONTHS_BALANCE']>=-3], 'SK_ID_CURR', agg_cols, agg_funcs='max')
Payments_Mon_3M_Max.columns = [f"{col}_3M_MAX" if col != 'SK_ID_CURR'  else col for col in Payments_Mon_3M_Max.columns]

Last Snapshot

In [111]:
Payments_Last_Snapshot= Payments_Mon.groupby(['SK_ID_CURR'])[agg_cols + ['MONTHS_BALANCE']].last().reset_index()
Payments_Last_Snapshot.columns = [col + 'LAST_MONTH' if col != 'SK_ID_CURR' else col for col in Payments_Last_Snapshot.columns]
Payments_Last_Snapshot.rename(columns={'MONTHS_BALANCE':'PMNT_INSTALMENT_DAYS_SINCE_LAST_TRANSACTION'}, inplace=True)

Merge all

In [112]:
Payments_Agg_final = Payments_Agg_PrevID_counts.merge(Payments_Agg_PrevID_Last1Y_counts, on='SK_ID_CURR', how='left')
Payments_Agg_final = Payments_Agg_final.merge(Payments_Mon_1Y_Avg, on='SK_ID_CURR', how='left')
Payments_Agg_final = Payments_Agg_final.merge(Payments_Mon_1Y_Max, on='SK_ID_CURR', how='left')
Payments_Agg_final = Payments_Agg_final.merge(Payments_Mon_3M_Avg, on='SK_ID_CURR', how='left')
Payments_Agg_final = Payments_Agg_final.merge(Payments_Mon_3M_Max, on='SK_ID_CURR', how='left')
Payments_Agg_final = Payments_Agg_final.merge(Payments_Last_Snapshot, on='SK_ID_CURR', how='left')

In [113]:
Payments_Agg_final = Payments_Agg_final.merge(base, on='SK_ID_CURR', how='left')

In [59]:
Payments_Agg_final.to_parquet("a:\\Dominic\\My Projects\\Home Credit\\src\\intermediate\\payment_installments_processed.parquet")